# System D — Stacked logit ensemble (beginner version)

**Goal:** combine several rating systems into one best-calibrated probability model.

Why you do this:
- Each system is good in different regimes.
  - Elo can be strong and stable on dense MMA data.
  - Glicko-2 handles inactivity and uncertainty better.
  - Dynamic factor model handles cross-sport and time variation.
- A meta-model can learn how to weight them to minimize prediction error.

---

## 1) Inputs

For an MMA bout, each base system outputs a win probability:

- $p_A$: System A (Elo-PS)
- $p_B$: System B (Glicko-2-PS)
- $p_C$: System C (dynamic factors)
- optional $p_N$: network/PageRank proxy (often weaker alone)

---

## 2) Stacking on logits (a simple, strong approach)

Probabilities live in $[0,1]$, but additive models work better on the **logit** scale:

$$
\text{logit}(p) = \log\left(\frac{p}{1-p}\right)
$$

A common stacking model is:

$$
\text{logit}(p_{\text{final}}) =
\beta_0 +
\beta_A\,\text{logit}(p_A) +
\beta_B\,\text{logit}(p_B) +
\beta_C\,\text{logit}(p_C) +
\gamma^\top f(b)
$$

where $f(b)$ can include:
- tier flag (A/B/C),
- title fight flag,
- scheduled rounds (3 vs 5),
- time since last fight,
- missingness indicators, etc.

---

## 3) Training (production rule)

To avoid lookahead bias:
- train using a **rolling-origin** split by time,
- validate on future windows,
- choose hyperparameters by log loss / Brier / calibration error.

This notebook uses synthetic data to show the mechanics.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

system_d_module = import_module('elo_calculator.application.ranking.system_d_stacked_logit')
types_module = import_module('elo_calculator.application.ranking.types')
StackedLogitMixtureSystem = system_d_module.StackedLogitMixtureSystem
EvidenceTier = types_module.EvidenceTier
StackingSample = types_module.StackingSample

## 4) Synthetic demo dataset

We generate:
- a hidden "true" probability
- three base probabilities $p_A, p_B, p_C$ that are noisy versions of truth
- outcomes sampled from the true probability

Then we fit the stacked logit model and compare log loss.


In [ ]:
rng = np.random.default_rng(7)
sample_count = 2500
latent_gap = rng.normal(0.0, 1.0, size=sample_count)

prob_true = 1.0 / (1.0 + np.exp(-1.2 * latent_gap))
prob_a = 1.0 / (1.0 + np.exp(-(1.0 * latent_gap + rng.normal(0.0, 0.65, size=sample_count))))
prob_b = 1.0 / (1.0 + np.exp(-(0.9 * latent_gap + rng.normal(0.0, 0.55, size=sample_count))))
prob_c = 1.0 / (1.0 + np.exp(-(1.3 * latent_gap + rng.normal(0.0, 0.70, size=sample_count))))
outcome = rng.binomial(1, prob_true, size=sample_count)

tiers = rng.choice(['A', 'B', 'C'], size=sample_count, p=[0.5, 0.2, 0.3])

samples = []
for idx in range(sample_count):
    tier = EvidenceTier(tiers[idx])
    sample = StackingSample(
        p_a=float(prob_a[idx]),
        p_b=float(prob_b[idx]),
        p_c=float(prob_c[idx]),
        p_n=None,
        outcome_a_win=float(outcome[idx]),
        tier=tier,
        scheduled_rounds=5 if idx % 5 == 0 else 3,
        is_title_fight=(idx % 11 == 0),
        days_since_last_mma=float(rng.integers(7, 600)),
        missing_weight_class=(idx % 17 == 0),
        missing_promotion=(idx % 23 == 0),
        opponent_connectivity=float(rng.integers(2, 40)),
    )
    samples.append(sample)

## 5) Fit a stacking model

We fit logistic regression on logits + a one-hot tier indicator.


In [ ]:
cutoff = int(sample_count * 0.8)
train_samples = samples[:cutoff]
test_samples = samples[cutoff:]

system = StackedLogitMixtureSystem()
system.fit(train_samples)


def binary_log_loss(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    clipped = np.clip(y_prob, 1e-9, 1.0 - 1e-9)
    return float(-np.mean(y_true * np.log(clipped) + (1.0 - y_true) * np.log(1.0 - clipped)))


def brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    return float(np.mean((y_true - y_prob) ** 2))


y_true = np.array([sample.outcome_a_win for sample in test_samples], dtype=np.float64)
y_prob_a = np.array([sample.p_a for sample in test_samples], dtype=np.float64)
y_prob_b = np.array([sample.p_b for sample in test_samples], dtype=np.float64)
y_prob_c = np.array([sample.p_c for sample in test_samples], dtype=np.float64)
y_prob_stack = np.array(system.predict_probabilities(test_samples), dtype=np.float64)

metrics = pd.DataFrame(
    [
        {'model': 'system_a', 'log_loss': binary_log_loss(y_true, y_prob_a), 'brier': brier_score(y_true, y_prob_a)},
        {'model': 'system_b', 'log_loss': binary_log_loss(y_true, y_prob_b), 'brier': brier_score(y_true, y_prob_b)},
        {'model': 'system_c', 'log_loss': binary_log_loss(y_true, y_prob_c), 'brier': brier_score(y_true, y_prob_c)},
        {
            'model': 'system_d_stack',
            'log_loss': binary_log_loss(y_true, y_prob_stack),
            'brier': brier_score(y_true, y_prob_stack),
        },
    ]
)

metrics.sort_values('log_loss').reset_index(drop=True)

## 6) What to take away

- Stacking is a **supervised calibration layer**.
- It doesn't replace base systems; it *uses them as features*.
- In production you fit it with a **time-based rolling origin** split.

Once you have `p_final`, you can create a stable leaderboard using:
- Expected Win Rate (System E), or
- convert `p_final` into a BT-style “worth” parameter via a fitted fighter-effect model.
